<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pruning_Rebuild.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — mount Drive + install packages
from google.colab import drive
drive.mount('/content/drive')

!pip -q install torch-pruning thop onnx onnxruntime

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 97.1 MB/s eta 0:00:00


In [2]:
# Cell 2 — fixed
import sys, shutil, os

UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"

if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

✅ utils loaded from Drive


In [3]:
# Cell 3 — prepare dataset (same as all your other notebooks)
from utils.dataset import prepare_dataset
from utils.train   import setup_device

device = setup_device(seed=74)
prepare_dataset()

Device: cuda
1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests


PosixPath('/content/vww_work/extracted/vw_coco2014_96')

In [4]:
# Cell 4 — rebuild + fine-tune (skips if checkpoint exists)
from utils.structured_rebuild import rebuild_and_measure
rebuild_and_measure(ratios=(0.02, 0.03), ft_epochs=15)

Device: cuda
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation

DENSE baseline   | params   139,428 | MACs   20,232,040 | test 79.13%
------------------------------------------------------------------------------
mv3_rebuilt_2pct  ✅ checkpoint found — skipping training
STRUCTURED 2%   | params   132,662 (-4.85%) | MACs   19,114,566 (-5.52%)
   acc pre-FT —%  →  post-FT 78.07%
   pred INT8 latency ~3.457 ms  (baseline 3.659 ms, noise 3.659–3.668 ms)
   predicted speedup 0.202 ms vs noise 0.009 ms → potentially measurable
------------------------------------------------------------------------------
mv3_rebuilt_3pct  ✅ checkpoint found — skipping training
STRUCTURED 3%   | params   130,006 (-6.76%) | MACs   18,767,143 (-7.24%)
   acc pre-FT —%  →  post-FT 79.6%
   pred INT8 latency ~3.394 ms  (baseline 3.659 ms, noise 3.659–3.668 ms)
   predicted speedup 0.265 ms vs noise 0.009 ms → potentially measurable
-------------------------------------------

In [5]:
# Cell 5 — ONNX export + INT8 + NSE (skips if already done)
from utils.structured_rebuild import quantize_rebuilt
quantize_rebuilt(ratios=(0.02, 0.03))

Device: cuda
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation
    ⏭️  Shared test files exist

mv3_rebuilt_2pct
    ✅ FP32 ONNX saved


    ✅ Calib data saved (200 samples)


    ✅ INT8 QDQ ONNX saved
  NSE 0.9891  →  ✅ go to board

mv3_rebuilt_3pct
    ✅ FP32 ONNX saved


    ✅ Calib data saved (200 samples)


    ✅ INT8 QDQ ONNX saved
  NSE 0.9912  →  ✅ go to board

✅ Records saved → /content/drive/My Drive/stm32-thesis/checkpoints/rebuild_records.json


In [6]:
# Cell 6 — view all saved results any time, no re-running
from utils.structured_rebuild import print_records
print_records()


── mv3_dense_baseline ──
   params                 139428
   macs                   20232040
   test_acc_%             79.13

── mv3_rebuilt_2pct ──
   ratio                  0.02
   params_pruned          132662
   macs_pruned            19114566
   param_cut_%            4.85
   mac_cut_%              5.52
   acc_pre_ft_%           —
   acc_post_ft_%          78.07
   pred_lat_ms            3.457
   nse                    0.9891
   nse_ok                 True

── mv3_rebuilt_3pct ──
   ratio                  0.03
   params_pruned          130006
   macs_pruned            18767143
   param_cut_%            6.76
   mac_cut_%              7.24
   acc_pre_ft_%           —
   acc_post_ft_%          79.6
   pred_lat_ms            3.394
   nse                    0.9912
   nse_ok                 True
